# Module 2: X-Ray Fracture Detection with DenseNet121

Binary classification pipeline: Fracture vs. No Fracture

- **Model**: DenseNet121 (ImageNet pretrained)
- **Datasets**: Combined Dataset A (predefined splits) + Dataset B (manual splits)
- **Training**: Two-phase fine-tuning with early stopping
- **Device**: CUDA if available, CPU fallback

In [ ]:
import os
import sys
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

# Set seaborn style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Import custom modules
from data_preprocessing import (
    get_combined_dataloader,
    show_batch,
    get_class_weights
)
from model import (
    build_densenet121,
    unfreeze_layers,
    get_model_summary
)
from train import (
    train_model,
    plot_training_history,
    load_model
)
from evaluate_module2 import (
    evaluate_on_test,
    predict_single_image
)

print(f"PyTorch version: {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# ============ Configuration ============
ROOT_A = './Bone_Fracture_Binary_Classification'
ROOT_B = './Bone fracture dataset'
BATCH_SIZE = 32
NUM_EPOCHS = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = './weights'

# Create weights directory
os.makedirs(SAVE_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True

print("="*70)
print("Configuration")
print("="*70)
print(f"Dataset A root: {ROOT_A}")
print(f"Dataset B root: {ROOT_B}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*70)

In [ ]:
# Load dataloaders for all phases
print("Loading dataloaders...")
train_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='train',
    batch_size=BATCH_SIZE
)
val_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='val',
    batch_size=BATCH_SIZE
)
test_loader = get_combined_dataloader(
    ROOT_A, ROOT_B,
    phase='test',
    batch_size=BATCH_SIZE
)

print("\n" + "="*70)
print("Dataset Information")
print("="*70)
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Total training samples: {len(train_loader) * BATCH_SIZE}")
print(f"Total validation samples: {len(val_loader) * BATCH_SIZE}")
print(f"Total test samples: {len(test_loader) * BATCH_SIZE}")
print("="*70)

# Visualize a batch
print("\nVisualizing training batch...")
show_batch(train_loader, n=8)

In [ ]:
# Build model
print("Building DenseNet121 model...")
model = build_densenet121(
    num_classes=2,
    pretrained=True,
    dropout_rate=0.5
)
model = model.to(DEVICE)

# Print model summary
get_model_summary(model)

print("\nModel architecture (first 5 layers):")
for i, (name, module) in enumerate(model.named_children()):
    if i < 5:
        print(f"  {name}: {module.__class__.__name__}")

In [ ]:
# Train the model
print("\nStarting training...\n")
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
    save_dir=SAVE_DIR,
    patience=7
)

print("\nTraining completed!")

In [ ]:
# Plot training curves
print("Plotting training history...")
plot_training_history(history)

# Print final metrics
print("\n" + "="*70)
print("Final Training Metrics")
print("="*70)
print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")
print(f"Final training loss: {history['train_loss'][-1]:.4f}")
print(f"Final validation loss: {history['val_loss'][-1]:.4f}")
print("="*70)

In [ ]:
# Load best model
best_model = load_model(
    checkpoint_path=os.path.join(SAVE_DIR, 'densenet121_best.pth'),
    device=DEVICE
)

# Evaluate on test set
evaluate_on_test(best_model, test_loader, DEVICE)

In [ ]:
# Example: Single image prediction
# Replace 'sample_xray.jpg' with actual image path
sample_image_path = './sample_xray.jpg'

# Check if sample image exists
if os.path.exists(sample_image_path):
    print(f"Predicting on: {sample_image_path}\n")
    result = predict_single_image(
        best_model,
        sample_image_path,
        device=DEVICE
    )
    
    print("="*70)
    print("Prediction Result")
    print("="*70)
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"\nProbabilities:")
    for class_name, prob in result['probabilities'].items():
        print(f"  {class_name}: {prob:.4f}")
    print("="*70)
else:
    print(f"Sample image not found at: {sample_image_path}")
    print("To test single image prediction, provide a valid image path.")
    print("\nExample usage:")
    print("  result = predict_single_image(best_model, 'path/to/image.jpg', device=DEVICE)")
    print("  print(result)")